In [ ]:
%run ./imports.py

## Paths

In [ ]:
prefix_minrtt_path = "data/campus_trace_prefixes_filtered_on_minrtt.csv"
countries_naturalearth_path = "data/countries_naturalearth.csv"
max_intra_hull_df_path = "data/max_intra_hull_df.pkl"
max_intra_hull_mainland_df_path = "data/max_intra_hull_mainland_df.pkl"
min_inter_hull_df_path = "data/min_inter_hull_df.pkl"
min_inter_hull_mainland_df_path = "data/min_inter_hull_mainland_df.pkl"
min_dev_hull_df_path = "data/min_dev_hull_df.pkl"
min_dev_hull_mainland_df_path = "data/min_dev_hull_mainland_df.pkl"

## Utility functions

In [ ]:
def compare_base_and_filtered(df_base, df_filtered):
    prefixes_before = df_base.shape[0]
    prefixes_after  = df_filtered.shape[0]

    bytes_before = df_base['Prefix_Total_Bytes'].sum()
    bytes_after  = df_filtered['Prefix_Total_Bytes'].sum()

    print(f"#Prefixes: {prefixes_after} ({round(prefixes_after*100/prefixes_before, 2)}%)")
    print(f"#Bytes: {bytes_after} ({round(bytes_after*100/bytes_before, 2)}%)")

In [ ]:
C_OPTICAL_FIBER_KM_PER_MS = (2/3) * (299792458 / 10**6)
print(f"Speed of light in optical fiber: {round(C_OPTICAL_FIBER_KM_PER_MS, 2)} km/ms")

## Load prefixes data

In [ ]:
df_prefixes_minrtt = pd.read_csv(prefix_minrtt_path)
print(f"df_prefixes_minrtt_filtered length: {df_prefixes_minrtt.shape[0]}")

In [ ]:
df_prefixes_minrtt.head(n=2)

## Filter prefixes data

In [ ]:
df_prefixes_minrtt_filtered = df_prefixes_minrtt[
    (df_prefixes_minrtt['Prefix_Duration_s'] >= 60*30)
    & (df_prefixes_minrtt['RTT_Count'] >= 5*30) ]

compare_base_and_filtered(df_prefixes_minrtt, df_prefixes_minrtt_filtered)

## MinRTT in prefix vs. geodesic distance

In [ ]:
minrtts_measured = df_prefixes_minrtt_filtered['minRTT_ms'].tolist()
geodist_princeton = df_prefixes_minrtt_filtered['Geodesic_Distance_km'].tolist()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(geodist_princeton, minrtts_measured, color='blue', s=5)

# Labels and title
plt.xlabel('Distance between source \nand destination (km)')
plt.ylabel('Measured\nmin. RTT (ms)')
plt.xlim(-1000, 21000)
plt.grid(True)
plt.show()

## Distances before and during interception attacks
* Distance between source (s) and destination (d) = D_sd
* Round-trip distance before attack = 2 * D_sd
* Round-trip distance via attacker (a) = D_sd + D_sa + D_da

In [ ]:
def rtt_lb_from_distance(d):
    return d / C_OPTICAL_FIBER_KM_PER_MS

In [ ]:
def impact_of_attack_distances(D_sd, D_sa, D_da):
    rt_dist_before = 2 * D_sd
    rtt_lb_before  = rtt_lb_from_distance(rt_dist_before)
    rt_dist_during = D_sd + D_sa + D_da
    rtt_lb_during  = rtt_lb_from_distance(rt_dist_during)
    return rt_dist_before, rt_dist_during, rtt_lb_before, rtt_lb_during

In [ ]:
def impact_of_attack_geolocations(geo_s, geo_d, geo_a):
    D_sd = geodesic(geo_s, geo_d)
    D_sa = geodesic(geo_s, geo_a)
    D_da = geodesic(geo_d, geo_a)
    return impact_of_attack_distances(D_sd, D_sa, D_da)

* Before attack
    * Minimum distance between source and destination = 0 km
    * Maximum distance between source and destination = 20,037.5 km (max. distance between 2 points on the earth's surface)
* During attack
    * Closest attacker is on path between the source and the destination, adding 0 km to the round-trip distance.
    * Farthest attacker is on the opposite side of the earth, causing the round-trip distance to be the circumference of the equator, i.e., 40,075 km.

In [ ]:
distances    = []
rtt_before   = [] # Lower bound of RTT before attack
rtt_closest  = [] # Lower bound of RTT during attack from closest attacker
rtt_farthest = [] # Lower bound of RTT during attack from farthest attacker

for d in range(0, 20101, 100):
    distances.append(d)
    rtt_before.append(rtt_lb_from_distance(2*d))
    rtt_closest.append(rtt_lb_from_distance(2*d))
    rtt_farthest.append(rtt_lb_from_distance(40075))

# Plot the curves
plt.figure(figsize=(10, 6))
plt.plot(distances, rtt_before, label='Before attack', linestyle='-', color='darkgreen', linewidth=3)
plt.plot(distances, rtt_closest, label='Closest attacker', linestyle='--', color='red', linewidth=3, alpha=0.7)
plt.plot(distances, rtt_farthest, label='Farthest attacker', linestyle=':', color='red', linewidth=3, alpha=0.7)
plt.scatter(geodist_princeton, minrtts_measured, label='Measured on campus', color='blue', s=5)

# Shade area between curves
plt.fill_between(distances, rtt_closest, rtt_farthest, color='red', alpha=0.3)

# Labels and title
plt.xlabel('Distance between source\nand destination (km)')
plt.ylabel('Min. RTT (ms)')
plt.xlim(-1000, 21000)
plt.legend()
plt.grid(True)
plt.show()

## Prefixes impossible to detect

In [ ]:
max_rtt_lb = rtt_lb_from_distance(40075)
print(f"Max. lower-bound RTT possible before or during attack: {mu.rnd(max_rtt_lb, 1)} ms")

In [ ]:
df_prefixes_never_detected = df_prefixes_minrtt_filtered[ df_prefixes_minrtt_filtered['minRTT_ms'] >=  max_rtt_lb]
compare_base_and_filtered(df_prefixes_minrtt_filtered, df_prefixes_never_detected)

In [ ]:
geo_distances_never_detected = df_prefixes_never_detected['Geodesic_Distance_km'].tolist()
pu.cdf(geo_distances_never_detected, {"xlabel": "Distance between source\nand destination (km)", "title": "Prefixes never detected"})

## Difference between per-prefix lower-bound RTT and measured minimum RTT

In [ ]:
df_prefixes_minrtts = df_prefixes_minrtt_filtered[['minRTT_ms', 'minRTT_Lower_Bound_ms']].copy()
df_prefixes_minrtts['minRTT_difference_ms'] = df_prefixes_minrtts['minRTT_ms'] - df_prefixes_minrtts['minRTT_Lower_Bound_ms']
min_rtt_diffs = df_prefixes_minrtts['minRTT_difference_ms'].tolist()

In [ ]:
pu.cdf(min_rtt_diffs, {"xlabel": "Difference between measured\nand theoretical min. RTT (ms)"})

## Country-based analysis

### Determine maximum distance within each country

In [ ]:
df_maxdist = pd.read_pickle(max_intra_hull_df_path)
df_maxdist.head(n=1)

In [ ]:
df_maxdist_mainland = pd.read_pickle(max_intra_hull_mainland_df_path)
df_maxdist_mainland.head(n=1)

### Maximum intra-country distances

In [ ]:
max_dists_all    = df_maxdist['MaxDistance_km'].tolist()
max_dists_main   = df_maxdist_mainland['MaxDistance_km'].tolist()
max_minowds_all  = df_maxdist['MaxOWD_ms'].tolist()
max_minowds_main = df_maxdist_mainland['MaxOWD_ms'].tolist()

In [ ]:
print("All regions:")
print(f"\t Min: {mu.rnd(min(max_dists_all), 0)} km")
print(f"\t p25: {mu.pctile_rnd(max_dists_all, 25, 0)} km, {mu.pctile_rnd(max_minrtts_all, 25, 1)} ms")
print(f"\t p50: {mu.pctile_rnd(max_dists_all, 50, 0)} km, {mu.pctile_rnd(max_minrtts_all, 50, 1)} ms")
print(f"\t p75: {mu.pctile_rnd(max_dists_all, 75, 0)} km, {mu.pctile_rnd(max_minrtts_all, 75, 1)} ms")
whisker_dist = mu.pctile_rnd(max_dists_all, 75, 0) + 1.5 * (mu.pctile_rnd(max_dists_all, 75, 0) - mu.pctile_rnd(max_dists_all, 25, 0))
whisker_rtt  = mu.pctile_rnd(max_minrtts_all, 75, 1) + 1.5 * (mu.pctile_rnd(max_minrtts_all, 75, 1) - mu.pctile_rnd(max_minrtts_all, 25, 1))
print(f"\t Upper whisker: {mu.rnd(whisker_dist, 0)} km, {mu.rnd(whisker_rtt, 1)} ms")
print(f"\t Max: {mu.rnd(max(max_dists_all), 0)} km")

print("Mainland only:")
print(f"\t Min: {mu.rnd(min(max_dists_main), 0)} km")
print(f"\t p25: {mu.pctile_rnd(max_dists_main, 25, 0)} km, {mu.pctile_rnd(max_minrtts_main, 25, 1)} ms")
print(f"\t p50: {mu.pctile_rnd(max_dists_main, 50, 0)} km, {mu.pctile_rnd(max_minrtts_main, 50, 1)} ms")
print(f"\t p75: {mu.pctile_rnd(max_dists_main, 75, 0)} km, {mu.pctile_rnd(max_minrtts_main, 75, 1)} ms")
whisker_dist = mu.pctile_rnd(max_dists_main, 75, 0) + 1.5 * (mu.pctile_rnd(max_dists_main, 75, 0) - mu.pctile_rnd(max_dists_main, 25, 0))
whisker_rtt  = mu.pctile_rnd(max_minrtts_main, 75, 1) + 1.5 * (mu.pctile_rnd(max_minrtts_main, 75, 1) - mu.pctile_rnd(max_minrtts_main, 25, 1))
print(f"\t Upper whisker: {mu.rnd(whisker_dist, 0)} km, {mu.rnd(whisker_rtt, 1)} ms")
print(f"\t Max: {mu.rnd(max(max_dists_main), 0)} km")

In [ ]:
top5 = df_maxdist.nlargest(5, 'MaxDistance_km')
top5.head()

In [ ]:
top5 = df_maxdist_mainland.nlargest(5, 'MaxDistance_km')
top5.head()

### Minimum inter-country distances

In [ ]:
df_mindist = pd.read_pickle(min_inter_hull_df_path)
df_mindist.head(n=1)

In [ ]:
df_mindist_mainland = pd.read_pickle(min_inter_hull_mainland_df_path)
df_mindist_mainland.head(n=1)

In [ ]:
top5 = df_mindist.nlargest(5, 'MinDistance_km')
top5.head()

In [ ]:
top5 = df_mindist_mainland.nlargest(5, 'MinDistance_km')
top5.head()

### Comparison between maximum intra-country and minimum inter-country distances and OWDs

In [ ]:
ys = []
ys.append(df_maxdist['MaxDistance_km'].tolist())
ys.append(df_maxdist_mainland['MaxDistance_km'].tolist())
ys.append(df_mindist['MinDistance_km'].tolist())
ys.append(df_mindist_mainland['MinDistance_km'].tolist())

ys2 = []
ys2.append(df_maxdist['MaxOWD_ms'].tolist())
ys2.append(df_maxdist_mainland['MaxOWD_ms'].tolist())
ys2.append(df_mindist['MinOWD_ms'].tolist())
ys2.append(df_mindist_mainland['MinOWD_ms'].tolist())

pos = []
xtick_pos = []
for i in range(2):
    pos.append(i - 0.2)
    pos.append(i + 0.2)
    xtick_pos.append(i)

fig, ax = plt.subplots(figsize=(11, 7))
boxes = ax.boxplot(ys, positions=pos, widths=0.3, patch_artist=True)
sns_colors = cycle(sns.color_palette("pastel"))

props = {"legend_1": "Entire country", "legend_2": "Mainland only"}
colors = {props["legend_1"]: next(sns_colors), props["legend_2"]: next(sns_colors)}
methods = list(colors.keys())

for idx, box in enumerate(boxes['boxes']):
    method = methods[idx%len(methods)]
    box.set_facecolor(colors[method])

ax.set_xticks([0, 1])
ax.set_xticklabels(["Intra-Country\nMaximum", "Inter-Country\nMinimum"], fontsize=25)

legend_handles = [mpatches.Patch(color=colors[method], label=method) for method in methods]
ax.legend(handles=legend_handles, loc='upper left', title="Country Area", facecolor="white", framealpha=1)
ax.set_ylabel("Geodesic Distance (km)")

ax2 = ax.twinx()
y_min, y_max = ax.get_ylim()
ax2.set_ylim(y_min/C_OPTICAL_FIBER_KM_PER_MS, y_max/C_OPTICAL_FIBER_KM_PER_MS)
ax2.set_ylabel('One‐Way Delay\nat Speed-of-Light (ms)')
ax2.set_yticks([0, 25, 50, 75, 100])
ax2.grid(False)

plt.tight_layout()

plt.savefig("plots/min_vs_max_dist_owd.pdf", dpi=300)
plt.show()

## Defensible countries based on theoretical minimum RTT

In [ ]:
df_mindev = pd.read_pickle(min_dev_hull_df_path)
df_mindev.head(n=1)

In [ ]:
df_mindev_mainland = pd.read_pickle(min_dev_hull_mainland_df_path)
df_mindev_mainland.head(n=1)

In [ ]:
mindev_all = df_mindev["MinDevRTT_ms"].tolist()
mindev_mainland = df_mindev_mainland["MinDevRTT_ms"].tolist()

In [ ]:
pu.cdfs([mindev_all, mindev_mainland], {"curvelabels": ["Entire country", "Mainland only"],
                                        "xlabel": "Minimum RTT Deviation\nat Speed-of-Light (ms)",
                                        "ylabel": "CDF",
                                       })

### Coverage: Examples

#### Find countries with best and worst coverages

In [ ]:
countries = sorted(df_mindev_mainland['Country1'].unique())
country_coverage = []
for country in countries:
    preattack_country_max = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['PreAttack_ms'].max()
    postattack_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['PostAttack_ms'].tolist()
    defense_count = sum(1 for postattack in postattack_country if postattack > preattack_country_max)
    defense_pct = defense_count * 100.0 / len(postattack_country)
    country_coverage.append((country, defense_pct))
sorted_country_coverage = sorted(country_coverage, key=lambda c: c[1])
for country, coverage_at_max in sorted_country_coverage:
    print(f"{country}: {round(coverage_at_max, 2)}%")

In [ ]:
countries = ["Russia", "New Zealand"]
ys = [None] * 4
for c, country in enumerate(countries):
    preattack_rtts  = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['PreAttack_ms'].tolist()
    postattack_rtts = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['PostAttack_ms'].tolist()
    ys[c] = preattack_rtts
    ys[c+2] = postattack_rtts

pos = []
xtick_pos = []
for i in range(2):
    pos.append(i - 0.2)
    pos.append(i + 0.2)
    xtick_pos.append(i)

fig, ax = plt.subplots(figsize=(8, 6))
# boxes = ax.boxplot([[], [], [], []], positions=pos, widths=0.3, patch_artist=True)
boxes = ax.boxplot(ys, positions=pos, widths=0.3, patch_artist=True)
sns_colors = list(sns.color_palette("bright"))

props = {"legend_1": countries[0], "legend_2": countries[1]}
colors = {props["legend_1"]: sns_colors[2], props["legend_2"]: sns_colors[4]}
methods = list(colors.keys())

for idx, box in enumerate(boxes['boxes']):
    method = methods[idx%len(methods)]
    box.set_facecolor(colors[method])

ax.set_xticks([0, 1])
ax.set_xticklabels(["Pre-Attack", "Mid-Attack"], fontsize=25)

legend_handles = [mpatches.Patch(color=colors[method], label=method) for method in methods]
ax.legend(handles=legend_handles, loc='upper left', title="Victim Country", facecolor="white", framealpha=1)
ax.set_ylabel("RTT (ms)")
ax.set_xlabel("Phase of Optimal Attack")
ax.set_ylim(-10, 210)

plt.tight_layout()

plt.savefig("plots/examples_pre_mid.pdf", dpi=300)
plt.show()

In [ ]:
countries = ["Russia", "New Zealand"]
ys = []
for country in countries:
    mindev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['MinDevRTT_ms'].tolist()
    ys.append(mindev_country)
pu.cdfs(ys, {
                "figsize": (8, 6), "colors": (sns_colors[2], sns_colors[4]),
                "xlabel": "Minimum $\delta_{deviation}$ (ms)", "ylabel": "Attack Scenarios (%)",
                "curvelabels": countries, "legend_title": "Victim Country",
                "yticks": ([i/100 for i in range(0, 101, 20)], list(range(0, 101, 20)))
            })

In [ ]:
countries = ["Russia", "New Zealand"]
xs, ys = [], []
for country in countries:
    preattack_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['PreAttack_ms'].tolist()
    postattack_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['PostAttack_ms'].tolist()
    x, y = [], []
    for p in range(1, 101, 1):
        preattack_at_p = np.percentile(preattack_country, p)
        defense_count = sum(1 for postattack in postattack_country if postattack > preattack_at_p)
        defense_pct = defense_count * 100.0 / len(postattack_country)
        x.append(p)
        y.append(defense_pct)
    xs.append(x)
    ys.append(y)
print(f"Minimum % attack scenarios: {mu.rnd(min(min(y) for y in ys), 1)}")

pu.lineplots(xs, ys, {
        "figsize": (8, 6),
        "colors": (sns_colors[2], sns_colors[4]),
        "linestyles": ["-"],
        "loc": "lower left",
        "curvelabels": countries,
        "ylim": (48, 102),
        "xlabel": "Pre-Attack MinRTT (%ile)",
        "ylabel": "Attack Scenarios (%)",
        "legend_title": "Victim Country",
        "plot_path": "plots/examples_cvgatc_pre.pdf"
})

In [ ]:
print(ys[1])

In [ ]:
countries = df_mindev_mainland['Country1'].unique()
total = 0
coverage = 0
coverages = []
for country in countries:
    rtts_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','PostAttack_ms']].apply(tuple, axis=1).tolist()
    defense_count = sum(1 for pre, post in rtts_country if post - pre >= 5)
    coverages.append((country, defense_count*100.0/len(rtts_country)))
    coverage += defense_count
    total += len(rtts_country)
    # print(f"{country}: {round(defense_pct, 1)}%")
coverage_pct = coverage * 100.0 / total
print(f"Overall coverage: {round(coverage_pct, 1)}%")
for country, coverage in sorted(coverages, key=lambda x: x[1]):
    print(country, round(coverage,1))

### Coverage: Speed-of-light

In [ ]:
countries = sorted(df_mindev_mainland["Country1"].unique())
preattacks = df_mindev_mainland["PreAttack_ms"].tolist()
postattacks = df_mindev_mainland["PostAttack_ms"].tolist()

x, y = [], []
for p in range(1, 101, 1):
    preattack_at_p = np.percentile(preattacks, p)
    defense_count = sum(1 for postattack in postattacks if postattack > preattack_at_p)
    defense_pct = defense_count * 100.0 / len(postattacks)
    x.append(p)
    y.append(defense_pct)

for i, j in zip(x, y):
    if round(j) >= 80 and round(j) <= 95:
        print(f"{round(j, 1)}% attacks > {i}%ile pre-attack RTT")

pu.lineplot(x, y, {
        "figsize": (8, 6), "colors": [sns_colors[3]],
        "ylim": (48, 102),
        "xlabel": "Pre-Attack MinRTT (%ile)",
        "ylabel": "Attack Scenarios (%)",
        "plot_path": "plots/cvgatc_pre.pdf"
})

In [ ]:
mindevs = df_mindev_mainland["MinDevRTT_ms"].tolist()
print(min(mindevs), max(mindevs))

x, y = [], []
for dev in range(0, 201, 1):
    defense_count = sum(1 for mindev in mindevs if mindev > dev)
    defense_pct = defense_count * 100.0 / len(mindevs)
    x.append(dev)
    y.append(defense_pct)

# for i, j in zip(x, y):
#     if round(j) >= 80 and round(j) <= 95:
#         print(f"{round(j, 1)}% attacks cause >= {i} ms deviation")

pu.lineplot(x, y, {
        "figsize": (8, 6), "colors": ["Blue"],
        "xlabel": "Minimum $\delta_{deviation}$ (ms)",
        "ylabel": "Attack Scenarios (%)",
        "plot_path": "plots/cvgatc_mindev.pdf"
})

### Median deviation

In [ ]:
countries = sorted(df_mindev_mainland['Country1'].unique())
deviations = []
maxd = 0
mind = 10000
for country in countries:
    mindevs = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['MinDevRTT_ms'].tolist()
    mindevs_p50 = np.percentile(mindevs, 50)
    deviations.append((country, mindevs_p50))
    if max(mindevs) > maxd:
        maxd = max(mindevs)
    if min(mindevs) < mind:
        mind = min(mindevs)
deviations = sorted(deviations, key=lambda d: d[1])
# for country, dev in deviations:
#     print(country, dev)
print(mind, maxd)

### Minimum deviation as percentage of pre-attack RTT

In [ ]:
countries = ["Russia", "New Zealand"]
xs, ys = [], []
for country in countries:
    coverage, times = [], []
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','MinDevRTT_ms']].apply(tuple, axis=1).tolist()
    for t in range(0, 201, 1):
        count = 0
        for pre, dev in dev_country:
            if dev >= 5:
                if dev/pre >= t:
                    count += 1
        times.append(t)
        coverage.append(count*100.0/len(dev_country))
        if count == 0:
            break
    ys.append(times)
    xs.append(coverage)

reload(pu)
pu.lineplots(xs, ys, {
        "figsize": (8, 6),
        "colors": (sns_colors[2], sns_colors[4]),
        "linestyles": ["-"],
        "loc": "upper right",
        "curvelabels": countries,
        "ylabel": "$\delta_{deviation}^*/\delta_{pre}^*$ (times)",
        "xlabel": "Optimal Attacks (%)",
        "legend_title": "Victim Country",
        "plot_path": "plots/examples_times_vs_attacks.pdf"
})

In [ ]:
countries = sorted(df_mindev_mainland['Country1'].unique())
dev_fracs = []
for country in countries:
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','MinDevRTT_ms']].apply(tuple, axis=1).tolist()
    for pre, dev in dev_country:
        if dev >= 5:
            pre = max(pre, 1)
            dev_fracs.append(dev/pre)

x, y = [], []
for t in range(0, 10001, 1):
    count = sum(1 for frac in dev_fracs if frac >= t)
    y.append(t)
    x.append(count * 100.0/(258 * 257))
    if count == 0:
        break

pu.lineplot(x, y, {
        "figsize": (8, 6),
        "colors": [sns_colors[3]],
        "linestyles": ["-"],
        "loc": "upper right",
        "xlim": (-5, 105),
        "ylabel": "$\delta_{deviation}^*/\delta_{pre}^*$ (times)",
        "xlabel": "Optimal Attacks (%)",
        "plot_path": "plots/all_times_vs_attacks.pdf"
})

In [ ]:
countries = ["Russia", "New Zealand"]
xs, ys = [], []
for country in countries:
    coverage, times = [], []
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','PostAttack_ms']].apply(tuple, axis=1).tolist()
    for t in range(1, 201, 1):
        count = 0
        for pre, post in dev_country:
            if post - pre >= 5:
                if post/pre >= t:
                    count += 1
        times.append(t)
        coverage.append(count*100.0/len(dev_country))
        if count == 0:
            break
    ys.append(times)
    xs.append(coverage)

countries = sorted(df_mindev_mainland['Country1'].unique())
dev_fracs = []
for country in countries:
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','PostAttack_ms']].apply(tuple, axis=1).tolist()
    for pre, post in dev_country:
        if post - pre >= 5:
            pre = max(pre, 1)
            dev_fracs.append(post/pre)

x, y = [], []
for t in range(1, 10001, 1):
    count = sum(1 for frac in dev_fracs if frac >= t)
    y.append(t)
    x.append(count * 100.0/(258 * 257))
    if round(x[-1]) >= 85:
        print(x[-1], y[-1])
    if count == 0:
        break

xs.append(x)
ys.append(y)

pu.lineplots(xs, ys, {
        "figsize": (8, 6),
        "colors": [sns_colors[2], sns_colors[4], sns_colors[3]],
        "linestyles": ["-"],
        "loc": "upper right",
        "curvelabels": ["Russia", "New Zealand", "All Countries"],
        "ylabel": "$\\tau_{mid}^*/\\tau_{pre}^*$ (times)",
        "xlabel": "Optimal Attacks (%)",
        "legend_title": "Optimal Attacks on",
        "plot_path": "plots/coverage_vs_times.pdf"
})

In [ ]:
countries = ["Russia", "New Zealand"]
xs, ys = [], []
for country in countries:
    coverage, times = [], []
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','PostAttack_ms']].apply(tuple, axis=1).tolist()
    for t in range(5, 201, 1):
        count = 0
        for pre, post in dev_country:
            if post - pre >= 5:
                if post - pre >= t:
                    count += 1
        times.append(t)
        coverage.append(count*100.0/len(dev_country))
        if count == 0:
            break
    ys.append(times)
    xs.append(coverage)

countries = sorted(df_mindev_mainland['Country1'].unique())
devs = []
for country in countries:
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country][['PreAttack_ms','PostAttack_ms']].apply(tuple, axis=1).tolist()
    devs.extend([(max(pre, 1), post) for pre, post in dev_country if post - pre >= 5])

x, y = [], []
for t in range(5, 201, 1):
    count = 0
    for pre, post in devs:
        if post - pre >= t:
            count += 1
    y.append(t)
    x.append(count * 100.0/(258 * 257))
    if round(x[-1]) >= 85:
        print(x[-1], y[-1])
    if count == 0:
        break
xs.append(x)
ys.append(y)
        
pu.lineplots(xs, ys, {
        "figsize": (8, 6),
        "colors": [sns_colors[2], sns_colors[4], sns_colors[3]],
        "linestyles": ["-"],
        "loc": "upper right",
        "curvelabels": ["Russia", "New Zealand", "All Countries"],
        "ylabel": "$\\tau_{mid}^* - \\tau_{pre}^*$ (ms)",
        "xlabel": "Optimal Attacks (%)",
        "ylim": (-10, 310),
        "legend_title": "Optimal Attacks on",
        "plot_path": "plots/coverage_vs_deviation.pdf"
})

In [ ]:
X, Y, Z = [], [], []
xticks_pos = []
yticks_pos = []
x_25, y_25 = [], []

for i, x in enumerate(range(0, 5001, 10)):
    for j, y in enumerate(range(0, 16001, 10)):
        if x > 0:
            pre = (2 * x) / C_OPTICAL_FIBER_KM_PER_MS
            post = (x + (2 * y)) / C_OPTICAL_FIBER_KM_PER_MS
            z = max(post - pre, 0)
            if round(z) == 25:
                x_25.append(x)
                y_25.append(y)
        
            X.append(x)
            Y.append(y)
            Z.append(z)

        if x % 1000 == 0:
            xticks_pos.append(i)
        if y % 4000 == 0:
            yticks_pos.append(j)

xticks_pos = sorted(list(set(xticks_pos)))
yticks_pos = sorted(list(set(yticks_pos)))

reload(pu)
# pu.heatmap(X, Y, Z,
#            {
#                 "figsize": (8, 6), "color": "viridis_r",
#                 "label": "$\delta_{deviation}$ (ms)",
#                 "xlabel": "$\delta(S,D)$ ($\\times10^3$ km)",
#                 "ylabel": "$\\frac{\delta(S,A) + \delta(D,A)}{2}$ ($\\times10^3$ km)",
#                 "xticks": (xticks_pos, list(range(6))),
#                 "yticks": (yticks_pos, list(range(0, 17, 4)))
#             })
pu.heatmap(X, Y, Z,
           {
                "figsize": (8, 6), "palette": "Reds", "bounds": list(range(0, 151, 25)),
                "label": "$\\tau_{mid} - \\tau_{pre}$ (ms)",
                "xlabel": "$\delta(S,D)$ ($\\times10^3$ km)",
                "ylabel": "$\\frac{\delta(S,A) + \delta(D,A)}{2}$ ($\\times10^3$ km)",
                "xticks": (xticks_pos, list(range(6))),
                "yticks": (yticks_pos, list(range(0, 17, 4))),
                "plot_path": "plots/dist_vs_dev_heatmap.png"
            })

In [ ]:
pu.lineplot(x_25, y_25, {"color": "red"})
print((y_25[-1] - y_25[0])/(x_25[-1] - x_25[0]))

In [ ]:
x, y = [], []
for i in range(5001):
    j = 2500 + (0.5 * i)
    x.append(i)
    y.append(j)
    if i in [1090, 1872]:
        print(i, j)
pu.lineplot(x, y)

In [ ]:
dists_pre = df_mindev_mainland["Dist_SD_km"].tolist()
print(np.percentile(dists_pre, 85))
print(np.percentile(dists_pre, 95))

In [ ]:
pu.scatterplot([], [], {
        "figsize": (8, 6),
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "xlabel": "Countries (%)",
        "ylabel": "Optimal Attacks (%)"
})

In [ ]:
countries = sorted(df_mindev_mainland['Country1'].unique())
coverages = []
for country in countries:
    dev_country = df_mindev_mainland[df_mindev_mainland['Country1'] == country]['MinDevRTT_ms'].tolist()
    count = sum(1 for dev in dev_country if dev >= 5)
    coverages.append(count * 100.0 / len(dev_country))
coverages = sorted(coverages, reverse=True)

pct_countries = []
pct_attacks = []
for c in range(1, len(countries)+1):
    a = coverages[c-1]
    pct_countries.append(c * 100.0 / len(countries))
    pct_attacks.append(a)

sns_colors = list(sns.color_palette("bright"))
pu.lineplot(pct_countries, pct_attacks, {
        "figsize": (8, 6),
        "colors": [sns_colors[0]],
        "linestyles": ["-"],
        "loc": "upper right",
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "xlabel": "Countries (%)",
        "ylabel": "Optimal Attacks (%)",
        "plot_path": "plots/countries_vs_attacks.pdf"
})

In [ ]:
for i, j in zip(pct_countries, pct_attacks):
    print(f"Countries: {i}, Coverage: {j}")